In [ ]:
# !pip install xlrd

In [ ]:
import pandas as pd

mt_fire = pd.read_excel('./data/sanbul.xlsx', header=2)
mt_fire.isna().sum()

In [ ]:
mt_fire.info()

In [ ]:
# mt_fire['locsi'].unique()
mt_fire = mt_fire.rename(columns={'발생장소_시도':'시도'
})
mt_fire['일시'] = pd.to_datetime(mt_fire['발생일시_년'].astype(str)+'-'+mt_fire['발생일시_월'].astype(str)+'-'+mt_fire['발생일시_일'].astype(str)).dt.date

In [ ]:
mt_fire.head(5)

In [ ]:

mt_fire['산불발생'] = 1
mt_fire

In [ ]:
weather_df = pd.read_csv('./data/weather_preprocessed.csv')
weather_df

In [ ]:
weather_df.isna().sum()


In [ ]:
# weather_df['지점명'].unique()
print(mt_fire['시도'].unique())
print(weather_df['지점명'].unique())

In [ ]:
sido_to_locs = {

    '강원': [
        '속초','북춘천','철원','대관령','춘천','북강릉','강릉','동해',
        '원주','영월','울진','울릉도','인제','홍천','태백','정선군'
    ],

    '경기': [
        '동두천','파주','수원','양평','이천'
    ],

    '인천': [
        '백령도','인천','강화'
    ],

    '서울': [
        '서울'
    ],

    '충북': [
        '충주','청주','서청주','제천','보은'
    ],

    '충남': [
        '서산','홍성','천안','보령','부여','금산'
    ],

    '대전': [
        '대전'
    ],

    '세종': [
        '세종'
    ],

    '경북': [
        '추풍령','안동','상주','포항','봉화','영주','문경',
        '청송군','영덕','의성','구미','영천','경주시'
    ],

    '전북': [
        '군산','전주','부안','임실','정읍','남원','장수',
        '고창','고창군'
    ],

    '전남': [
        '목포','여수','흑산도','완도','순천',
        '진도(첨찰산)','보성군','강진군','장흥',
        '해남','고흥','영광군','진도군'
    ],

    '대구': [
        '대구'
    ],

    '울산': [
        '울산'
    ],

    '경남': [
        '창원','북창원','통영','진주','김해시','양산시',
        '의령군','함양군','거창','합천','밀양',
        '산청','거제','남해'
    ],

    '광주': [
        '광주'
    ],

    '부산': [
        '부산','북부산'
    ],

    '제주': [
        '제주','고산','성산','서귀포'
    ]
}


In [ ]:
loc_to_sido = {}
for sido,locs in sido_to_locs.items():
    # print(locs)
    for loc in locs:
        loc_to_sido[loc] = sido
        # print(loc_to_sido)


print(type(loc_to_sido))

In [ ]:
loc_to_sido

In [ ]:
weather_df['시도'] = weather_df['지점명'].map(loc_to_sido) # 속초 : 강원 --> 변환 map

# 매핑 안 된 지점은 '기타' 처리
weather_df['시도'] = weather_df['시도'].astype(str)

# 확인
print(weather_df[['지점명', '시도']].head())
weather_df

In [ ]:
weather_df['일시'] = pd.to_datetime(weather_df['일시']).dt.date
mt_fire['일시'] = pd.to_datetime(mt_fire['일시']).dt.date
weather_df = weather_df.dropna(subset=['시도'])

weather_sido = (
    weather_df
    .groupby(['일시', '시도'])
    .agg({
        '평균기온(°C)': 'mean',
        '최고기온(°C)': 'max',
        '일강수량(mm)': 'sum',
        '평균 상대습도(%)': 'mean',
        '최대 풍속(m/s)': 'max',
        '평균 풍속(m/s)': 'mean',
        '최저기온(°C)': 'min',
        '실효습도' : 'mean'
    })
    .reset_index()
)
weather_sido['일시'] = pd.to_datetime(weather_sido['일시'])
mt_fire['일시'] = pd.to_datetime(mt_fire['일시'])

print(weather_sido)
weather_loc_mtfire = pd.merge(weather_sido,mt_fire,how='inner',right_on=['일시','시도'], left_on=['일시','시도'])
weather_loc_mtfire


In [ ]:
fire_daily = (
    mt_fire
    .groupby(['시도','일시','발생장소_시군구'])
    .size()
    .reset_index(name='fire_count')
)

fire_daily['fire'] = 1
fire_daily = fire_daily[['시도','일시','fire','발생장소_시군구']]

fire_daily

In [ ]:
fire_daily['시도'].value_counts().sum()

In [ ]:
fire_daily

In [ ]:
# weather_sido = weather_sido.drop('지점명',axis=1)
# weather_sido

In [ ]:
fire_daily = fire_daily.drop('발생장소_시군구',axis=1)

In [ ]:
full_dates = pd.date_range(
    start=weather_loc_mtfire['일시'].min(),
    end=weather_loc_mtfire['일시'].max(),
    freq='D'
)

sido_list = weather_loc_mtfire['시도'].unique()
# print(sido_list)
base_df = (
    pd.MultiIndex
    .from_product([sido_list, full_dates], names=['시도','일시'])
    .to_frame(index=False)
)

print(base_df)
base_df = base_df.merge(
    fire_daily,
    on=['시도','일시'],
    how='left'
)

base_df['fire'] = base_df['fire'].fillna(0).astype(int)
base_df
base_df = base_df.merge(
    weather_sido,
    on=['시도', '일시'],
    how='left'
)

base_df


In [ ]:
base_df['fire'].value_counts()

In [ ]:
base_df.to_csv('머신러닝용_ver_0.1.csv',encoding='utf-8')

In [ ]:
weather_loc_mtfire

In [ ]:
full_dates = pd.date_range(
    start='2016-01-01',
    end='2025-12-31',
    freq='D'
)
sido_list = weather_loc_mtfire['시도'].unique()
full_index = pd.MultiIndex.from_product(
    [sido_list, full_dates],
    names=['시도', '일시']
)
weather_loc_mtfire['일시'] = pd.to_datetime(weather_loc_mtfire['일시'])
full_index


base_df = pd.DataFrame(index=full_index).reset_index()
base_df
base_df = pd.merge(
    weather_loc_mtfire,base_df,
    on=['시도', '일시'],
    how='right'
)
base_df['일시'] = pd.to_datetime(base_df['일시'])
base_df
base_df



In [ ]:
weather_loc_mtfire.to_csv('산불 날짜 별 날씨데이터.csv',index=False,encoding='utf-8')

In [84]:
weather_loc_mtfire.isnull().sum()

일시              0
시도              0
평균기온(°C)        0
최고기온(°C)        0
일강수량(mm)        0
평균 상대습도(%)      0
최대 풍속(m/s)      0
평균 풍속(m/s)      0
최저기온(°C)        0
실효습도            0
발생일시_년          0
발생일시_월          0
발생일시_일          0
발생일시_시간         0
발생일시_요일         0
진화종료시간_년        0
진화종료시간_월        0
진화종료시간_일        0
진화종료시간_시간       0
발생장소_관서         1
발생장소_시군구       12
발생장소_읍면       556
발생장소_동리       367
발생원인_구분       897
발생원인_세부원인       0
발생원인_기타         0
피해면적_합계         0
산불발생            0
dtype: int64